In [17]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
import pickle


In [18]:
def cargar_datos_preprocesados(ruta_csv: Path) -> pd.DataFrame:
    """Carga el dataset preprocesado con temperatura y rocío"""
    return pd.read_csv(ruta_csv)

In [19]:
df = cargar_datos_preprocesados(Path("data/NewData/skbo_ventana.csv"))

In [20]:
df.head()

,OACI,Aeropuerto,Latitud,Longitud,Fecha,METAR,FECHA_HORA_REPORTE
0,SKBO,Bogotá / El Dorado,4.70306,-74.13833,07-05-2026 13:05:00,SKBO 071300Z VRB04KT 9000 FEW015 SCT040 14/13 ...,2026-05-07 13:05:00
1,SKBO,Bogotá / El Dorado,4.70306,-74.13833,07-05-2026 13:05:00,SKBO 071315Z 07007KT 030V130 9999 FEW020 SCT04...,2026-05-07 13:05:00
2,SKBO,Bogotá / El Dorado,4.70306,-74.13833,07-05-2026 14:05:00,SKBO 071400Z 07006KT 020V110 9999 FEW020 BKN03...,2026-05-07 14:05:00
3,SKBO,Bogotá / El Dorado,4.70306,-74.13833,07-05-2026 15:05:00,SKBO 071500Z 21004KT 150V260 9999 BKN030 18/10...,2026-05-07 15:05:00
4,SKBO,Bogotá / El Dorado,4.70306,-74.13833,07-05-2026 16:05:00,SKBO 071600Z 09008KT 050V140 9999 BKN040 18/10...,2026-05-07 16:05:00


In [21]:
df_raw = df.copy()

In [22]:
# Eliminar registros inválidos que contienen "NIL"
registros_iniciales = len(df_raw)
invalid_count = df_raw["METAR"].str.contains("NIL", na=False).sum()
invalid_percent = (invalid_count / registros_iniciales) * 100
print(f"\n Registros iniciales: {registros_iniciales:,}")
print(f" Registros inválidos (NIL): {invalid_count:,} ({invalid_percent:.2f}%)")

# Eliminar registros inválidos
df_raw = df_raw[~df_raw["METAR"].str.contains("NIL", na=False)].copy()

registros_validos = len(df_raw)
print(f" Registros válidos después de limpieza: {registros_validos:,}")
print(f" Registros eliminados: {registros_iniciales - registros_validos:,}")

# Verificar forma después de limpieza
print(f"\n Forma del dataset después de limpieza: {df_raw.shape}")

print("=" * 60)
print("NORMALIZACIÓN DE FECHAS")
print("=" * 60)

df_raw["FECHA_HORA_REPORTE"] = pd.to_datetime(df_raw["FECHA_HORA_REPORTE"].astype(str), errors='coerce')

# Verificar conversión
fechas_invalidas = df_raw["FECHA_HORA_REPORTE"].isna().sum()
if fechas_invalidas > 0:
    print(f"  Advertencia: {fechas_invalidas} fechas no pudieron ser convertidas")

print(f" Columna FECHA_HORA_REPORTE creada")
print(f"   Rango: {df_raw['FECHA_HORA_REPORTE'].min()} a {df_raw['FECHA_HORA_REPORTE'].max()}")

# Extraer componentes temporales para uso posterior
df_raw["Año"] = df_raw["FECHA_HORA_REPORTE"].dt.year
df_raw["Mes"] = df_raw["FECHA_HORA_REPORTE"].dt.month
df_raw["Dia"] = df_raw["FECHA_HORA_REPORTE"].dt.day
df_raw["Hora"] = df_raw["FECHA_HORA_REPORTE"].dt.hour

# Eliminar registro con fecha 1900 si existe (error de datos)
if df_raw[df_raw['Año'] == 1900].shape[0] > 0:
    registros_eliminados_fecha = len(df_raw[df_raw['Año'] == 1900])
    df_raw = df_raw[df_raw['Año'] != 1900].copy()
    print(f" Eliminados {registros_eliminados_fecha} registros con fecha inválida (1900)")

print(f"\n Forma del dataset después de normalización: {df_raw.shape}")

# Verificar valores nulos en el dataframe completo y mostrar resumen
print("=" * 60)
print("VERIFICACIÓN DE VALORES NULOS")
print("=" * 60)
nulos_por_columna = df_raw.isnull().sum()
print(nulos_por_columna[nulos_por_columna > 0])

total_nulos = df_raw.isnull().sum().sum()
print(f"Total de valores nulos en el dataframe: {total_nulos}")

# Si se quiere enfocar en columnas clave:
columnas_clave = ["FECHA_HORA_REPORTE", "METAR"]
for col in columnas_clave:
    if col in df_raw.columns:
        nulos = df_raw[col].isnull().sum()
        print(f"{col}: {nulos} nulos")


# 1. Definir el patrón Regex
# \bAUTO\b busca la palabra exacta "AUTO"
# \s? busca un espacio opcional después de la palabra
patron_auto = r'\bAUTO\b\s?'

# 2. Aplicar la limpieza en la columna original
# Usamos regex=True para que interprete el patrón correctamente
df_raw["TEXTO_REPORTE_CLEAN"] = (
    df_raw["METAR"]
    .str.replace(patron_auto, "", regex=True, flags=re.IGNORECASE)
    .str.strip() # Elimina espacios al inicio o final si los hubiera
)

# 3. Tokenizar el texto ya limpio
df_raw["tokens"] = df_raw["TEXTO_REPORTE_CLEAN"].str.split()

print(f" Texto tokenizado")
print(f"   Ejemplo de tokens: {df_raw['tokens'].iloc[0][:5]}...")  # Mostrar primeros 5 tokens

df_raw[['METAR', 'tokens']].head(3)

# Borrar las filas donde el df_raw["tokens"].str[0] tenga una longitud diferente de 4 osea que el aerodromo fue mal diligenciado
print(f"Forma del DataFrame antes de la limpieza AERODROMO: {df_raw.shape}")

df_raw = df_raw[df_raw['tokens'].str[0].str.len() == 4].copy()

print(f"Forma del DataFrame después de la limpieza AERODROMO: {df_raw.shape}")

# Borrar las filas donde el df_raw["tokens"].str[1] tenga una longitud de 6 y 7 osea que la hora esta completa y tambien que le falto la Z al final
print(f"Forma del DataFrame antes de la limpieza HORA: {df_raw.shape}")

df_raw = df_raw[(df_raw['tokens'].str[1].str.len() == 6) | (df_raw['tokens'].str[1].str.len() == 7)].copy()

print(f"Forma del DataFrame después de la limpieza HORA: {df_raw.shape}")

# Borrar las filas donde el df_raw["tokens"].str[2] tenga una longitud diferente del 6 al 10 osea que la hora esta completa y tambien que le falto la Z al final
print(f"Forma del DataFrame antes de la limpieza VIENTO: {df_raw.shape}")

df_raw = df_raw[df_raw['tokens'].str[2].str.len().isin([6,7,8,9,10])].copy()

print(f"Forma del DataFrame después de la limpieza VIENTO: {df_raw.shape}")

# Paso 2: Extracción de variables meteorológicas usando parsing estructurado
# Los reportes METAR siguen un formato estándar: AERODROMO FECHAZ VIENTO VISIBILIDAD TEMP/ROCIO PRESION NUBOSIDAD FENOMENOS

print("Extrayendo variables meteorológicas...")

# Variables básicas (por posición en el tokenizado)
df_raw["aerodromo"] = df_raw["tokens"].str[0]      # Primer token: código del aeródromo
df_raw["fecha_zulu"] = df_raw["tokens"].str[1]     # Segundo token: fecha Zulu
df_raw["viento"] = df_raw["tokens"].str[2]         # Tercer token: viento
# la temperatura y el rocio no están en el mismo token en todos los reportes, por lo que se hará una extracción más flexible

def encontrar_temperatura_rocio(tokens_series):
    """Busca el token que contiene temperatura/rocío en cualquier posición"""
    
    # Crear una función para aplicar a cada fila
    def buscar_temp_rocio(tokens_list):
            if tokens_list is None:
                return np.nan
                
            if isinstance(tokens_list, str):
                tokens_list = [tokens_list]
            elif not isinstance(tokens_list, (list, tuple, np.ndarray)):
                return np.nan
                
            if len(tokens_list) == 0:
                return np.nan
                
            for token in tokens_list:
                if token is not None:
                    token_str = str(token).strip()
                    
                    # Patrón 1: Buscar formato XX/YY o XX / YY o XX/ YY o XX /YY
                    patron_con_barra = r'(\d{1,3}(?:\.\d+)?)\s*/\s*(\d{1,3}(?:\.\d+)?)'
                    match = re.search(patron_con_barra, token_str)
                    
                    if match:
                        temp_part = match.group(1)
                        rocio_part = match.group(2)
                        return f"{temp_part}/{rocio_part}"
            
            return np.nan
    
    return tokens_series.apply(buscar_temp_rocio)
    


# Aplicar la función mejorada
df_raw["temperatura_rocio"] = encontrar_temperatura_rocio(df_raw["tokens"])
df_raw['temperatura'] = df_raw['temperatura_rocio'].str.split('/').str[0]
df_raw['rocio'] = df_raw['temperatura_rocio'].str.split('/').str[1]


df_preprocesado = df_raw[["FECHA_HORA_REPORTE","METAR",'Año', 'Mes', 'Dia', 'Hora','viento', 'temperatura', 'rocio']].copy()
print(df_preprocesado.head())


 Registros iniciales: 60
 Registros inválidos (NIL): 0 (0.00%)
 Registros válidos después de limpieza: 60
 Registros eliminados: 0

 Forma del dataset después de limpieza: (60, 7)
NORMALIZACIÓN DE FECHAS
 Columna FECHA_HORA_REPORTE creada
   Rango: 2026-05-07 13:05:00 a 2026-05-09 15:05:00

 Forma del dataset después de normalización: (60, 11)
VERIFICACIÓN DE VALORES NULOS
Series([], dtype: int64)
Total de valores nulos en el dataframe: 0
FECHA_HORA_REPORTE: 0 nulos
METAR: 0 nulos
 Texto tokenizado
   Ejemplo de tokens: ['SKBO', '071300Z', 'VRB04KT', '9000', 'FEW015']...
Forma del DataFrame antes de la limpieza AERODROMO: (60, 13)
Forma del DataFrame después de la limpieza AERODROMO: (60, 13)
Forma del DataFrame antes de la limpieza HORA: (60, 13)
Forma del DataFrame después de la limpieza HORA: (60, 13)
Forma del DataFrame antes de la limpieza VIENTO: (60, 13)
Forma del DataFrame después de la limpieza VIENTO: (60, 13)
Extrayendo variables meteorológicas...
   FECHA_HORA_REPORTE     

In [23]:
df_pre_copy = df_preprocesado.copy()

In [24]:
#contar el numero de registros donde las columnas de temperatura y punto de rocío son nulas vs cantidad de registros totales
total_registros = len(df_pre_copy)
nulos_temperatura = df_pre_copy['temperatura'].isnull().sum()
nulos_rocio = df_pre_copy['rocio'].isnull().sum()
print(f"Total de registros: {total_registros}")
print(f"Registros con temperatura nula: {nulos_temperatura} ({(nulos_temperatura / total_registros) * 100:.2f}%)")
print(f"Registros con punto de rocío nulo: {nulos_rocio} ({(nulos_rocio / total_registros) * 100:.2f}%)")
#mostrar registros con temperatura nula pero con punto de rocío no nulo
diferencias =df_pre_copy[(df_pre_copy['temperatura'].isnull()) & (df_pre_copy['rocio'].notnull())]
print(f"Registros con temperatura nula pero punto de rocío no nulo: {len(diferencias)}")
df_pre_copy[df_pre_copy['temperatura'].isnull()].head()

Total de registros: 60
Registros con temperatura nula: 0 (0.00%)
Registros con punto de rocío nulo: 0 (0.00%)
Registros con temperatura nula pero punto de rocío no nulo: 0


,FECHA_HORA_REPORTE,METAR,Año,Mes,Dia,Hora,viento,temperatura,rocio


In [25]:
df_pre_copy.head()

,FECHA_HORA_REPORTE,METAR,Año,Mes,Dia,Hora,viento,temperatura,rocio
0,2026-05-07 13:05:00,SKBO 071300Z VRB04KT 9000 FEW015 SCT040 14/13 ...,2026,5,7,13,VRB04KT,14,13
1,2026-05-07 13:05:00,SKBO 071315Z 07007KT 030V130 9999 FEW020 SCT04...,2026,5,7,13,07007KT,16,12
2,2026-05-07 14:05:00,SKBO 071400Z 07006KT 020V110 9999 FEW020 BKN03...,2026,5,7,14,07006KT,16,11
3,2026-05-07 15:05:00,SKBO 071500Z 21004KT 150V260 9999 BKN030 18/10...,2026,5,7,15,21004KT,18,10
4,2026-05-07 16:05:00,SKBO 071600Z 09008KT 050V140 9999 BKN040 18/10...,2026,5,7,16,09008KT,18,10


In [26]:
# Convertir FECHA_HORA_REPORTE a datetime y establecer como índice
df_pre_copy['FECHA_REPORTE'] = pd.to_datetime(df_pre_copy['FECHA_HORA_REPORTE'])
df_pre_copy.set_index('FECHA_REPORTE', inplace=True)

# Eliminación de variables no necesarias, solo me quedo con la columna de viento
df_pre_copy.drop(columns=['Año', 'Mes', 'Dia', 'Hora'], inplace=True)

# Eliminar valores nulos en la columna 'viento' antes de aplicar el parseo
df_pre_copy = df_pre_copy[df_pre_copy['viento'].notna()]

# Modificar el valor KKT, T, K  por KT en la columna de viento
df_pre_copy['viento'] = df_pre_copy['viento'].str.replace(r'[A-Za-z]{3}$', 'KT', regex=True)

def parse_viento_flexible(cadena):
    if pd.isna(cadena) or not isinstance(cadena, str):
        return pd.Series([None, None, None])

    patron = r'^(\d{3}|VRB)(\d{2,3})(G\d{2,3})?KT$'
    match = re.match(patron, cadena)

    if match:
        dir_raw = match.group(1)
        vel_raw = match.group(2)
        gus_raw = match.group(3)

        direccion = float(dir_raw) if dir_raw != 'VRB' else None
        velocidad = float(vel_raw)

        # CORRECCIÓN: Si no hay ráfaga (gus_raw es None), ponemos 0
        rafaga = float(gus_raw[1:]) if gus_raw else 0.0

        return pd.Series([direccion, velocidad, rafaga])

    return pd.Series([None, None, None])

# Aplicación Regex para parsear la columna de viento con formato flexible
df_pre_copy[['direccion', 'intensidad_kt', 'rafaga_kt']] = df_pre_copy['viento'].apply(parse_viento_flexible)

# Toma el último grado válido (ej. 030) y lo pone donde estaba el NaN de 'VRB'
df_pre_copy['direccion'] = df_pre_copy['direccion'].ffill()

# Elimino errores de digitación, por lo cual me quedo cuando la longitud de viento es mayor a 7
df_pre_copy = df_pre_copy[df_pre_copy['viento'].str.len() > 6]

# Elimino aquellos que no se identifica intensidad del viento
df_pre_copy = df_pre_copy[df_pre_copy["intensidad_kt"].notnull()]

# Verificar cuantos no tienen la estructura
df_pre_copy["viento"].str.len().value_counts()

# Eliminar valores donde la dirección es muy desfazada
df_pre_copy = df_pre_copy[df_pre_copy["direccion"] <= 360]

In [29]:
# 1. Crear el horario maestro (ejemplo: cada 1 hora)
rango_teorico = pd.date_range(start=df_pre_copy.index.min().floor('h'),
                              end=df_pre_copy.index.max().ceil('h'),
                              freq='1h')

# 2. Identificar qué horas del maestro no están en tus datos
horas_faltantes = rango_teorico.difference(df_pre_copy.index)
print(f"Total de reportes faltantes: {len(horas_faltantes)}")

# Identificar reportes que no ocurren exactamente en la hora (minuto != 0)
reportes_especiales = df_pre_copy[df_pre_copy.index.minute != 0]
print(f"Reportes extra encontrados: {len(reportes_especiales)}")

Total de reportes faltantes: 52
Reportes extra encontrados: 59


In [30]:
# =========================================================
# FASE 1: PREPARACIÓN DEL ÍNDICE Y CÁLCULO DE COMPONENTES
# =========================================================

# Asegurarnos de que el índice de dataset original es formato datetime
df_pre_copy.index = pd.to_datetime(df_pre_copy.index)

# PASO CRÍTICO: Redondear los reportes a la hora en punto más cercana.
# Esto sincroniza reportes emitidos al minuto 50-59 con la hora exacta.
df_pre_copy.index = df_pre_copy.index.round('h')

# Al redondear, un reporte regular y un reporte especial (SPECI) podrían caer
# en la misma hora. Conservamos el último (el más actualizado).
df_pre_copy = df_pre_copy[~df_pre_copy.index.duplicated(keep='last')]

# Calcular u y v desde cero:
df_pre_copy['dir_sin'] = np.sin(np.radians(df_pre_copy['direccion'])) 
df_pre_copy['dir_cos'] = np.cos(np.radians(df_pre_copy['direccion']))

# Transformación logaritmica para intensidad
df_pre_copy['intensidad_log'] = np.log1p(df_pre_copy['intensidad_kt'])  # log(1 + x) para manejar ceros

In [31]:
# 1. Transformación Vectorial (u, v) - HACER ANTES DEL RESAMPLE

cols_numericas = ['dir_sin', 'dir_cos', 'intensidad_log', 'rafaga_kt', 'temperatura', 'rocio']
for col in cols_numericas:
    df_pre_copy[col] = pd.to_numeric(df_pre_copy[col], errors='coerce')
    
# 2. Definir reglas de agregación para tus columnas
reglas = {
    'dir_sin': 'mean',          # Promedio de la componente zonal
    'dir_cos': 'mean',          # Promedio de la componente meridional
    'intensidad_log': 'mean',
    'rafaga_kt': 'max',    # Capturamos la ráfaga más fuerte de la hora
    'temperatura': 'mean', # Promedio de temperatura
    'rocio': 'mean'       # Promedio de punto de rocío
}

# 3. Forzar la frecuencia horaria (Resample)
df_modelo = df_pre_copy.resample('1h').agg(reglas)

# 4. Mantener la continuidad para el modelo (Interpolación)
# El LSTM requiere que no existan huecos en la secuencia
df_modelo['dir_sin'] = df_modelo['dir_sin'].interpolate(method='linear')
df_modelo['dir_cos'] = df_modelo['dir_cos'].interpolate(method='linear')
df_modelo['intensidad_log'] = df_modelo['intensidad_log'].interpolate(method='linear')
df_modelo['rafaga_kt'] = df_modelo['rafaga_kt'].fillna(0) # Si no hay reporte, rafaga es 0
# Conservar las columnas 'temperatura' y 'rocio' en df_modelo
df_modelo['temperatura'] = df_modelo['temperatura'].interpolate(method='linear')
df_modelo['rocio'] = df_modelo['rocio'].interpolate(method='linear')

# Muestra la sumatoria de valores NaN en cada columna
print("Cantidad de valores faltantes por columna:")
print(df_modelo.isnull().sum())

# Verifica si la frecuencia horaria es estrictamente regular
print("\n¿Es el índice estrictamente regular (1H)?:", pd.infer_freq(df_modelo.index) == 'h')

# 1. Asignar la frecuencia horaria de forma estricta
df_modelo = df_modelo.asfreq('h')

# 2. Verificar que la etiqueta 'freq' se haya asignado correctamente
print("Frecuencia oficial del índice:", df_modelo.index.freq)

Cantidad de valores faltantes por columna:
dir_sin           0
dir_cos           0
intensidad_log    0
rafaga_kt         0
temperatura       0
rocio             0
dtype: int64

¿Es el índice estrictamente regular (1H)?: True
Frecuencia oficial del índice: <Hour>


In [33]:
df_modelo.head()

,dir_sin,dir_cos,intensidad_log,rafaga_kt,temperatura,rocio
FECHA_REPORTE,,,,,,
2026-05-07 13:00:00,0.939693,3.420201e-01,2.079442,0.0,16.0,12.0
2026-05-07 14:00:00,0.939693,3.420201e-01,1.945910,0.0,16.0,11.0
2026-05-07 15:00:00,-0.500000,-8.660254e-01,1.609438,0.0,18.0,10.0
2026-05-07 16:00:00,1.000000,6.123234e-17,2.197225,0.0,18.0,10.0
2026-05-07 17:00:00,0.866025,-5.000000e-01,2.079442,0.0,18.0,10.0


In [40]:
df_lstm = df_modelo.copy()

features_modelo = ['dir_sin', 'dir_cos', 'intensidad_log', 'temperatura', 'rocio']
X = df_lstm[features_modelo].values

# Lo que el modelo va a predecir
targets = ['dir_sin', 'dir_cos', 'intensidad_log']
y = df_lstm[targets].values

# ========== 1. NORMALIZAR VARIABLES ==========
print("Normalizando variables...")

# Escalar variables de entrada
scaler_X = StandardScaler()
df_lstm[features_modelo] = scaler_X.fit_transform(df_lstm[features_modelo])

# Escalar variables de salida
scaler_y = StandardScaler()
df_lstm[targets] = scaler_y.fit_transform(df_lstm[targets])

Normalizando variables...


In [41]:
df_lstm.head()

,dir_sin,dir_cos,intensidad_log,rafaga_kt,temperatura,rocio
FECHA_REPORTE,,,,,,
2026-05-07 13:00:00,0.900926,-0.052448,0.991470,0.0,0.343000,0.919475
2026-05-07 14:00:00,0.900926,-0.052448,0.684115,0.0,0.343000,0.361222
2026-05-07 15:00:00,-1.188204,-2.309510,-0.090356,0.0,0.881245,-0.197030
2026-05-07 16:00:00,0.988438,-0.691464,1.262576,0.0,0.881245,-0.197030
2026-05-07 17:00:00,0.794028,-1.625643,0.991470,0.0,0.881245,-0.197030
